<a href="https://colab.research.google.com/github/Fatuma23/JAVASCRIPT-TUTORIAL/blob/main/economic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install faker pandas numpy

In [19]:
%%writefile generate_data.py
import pandas as pd
import numpy as np
from faker import Faker
import argparse
import os

fake = Faker()

def generate_data(rows, seed, out):
    Faker.seed(seed)
    np.random.seed(seed)

    # Customers
    customers = []
    for i in range(rows):
        customers.append({
            "customer_id": f"C{i}",
            "customer_tier": np.random.choice(["bronze","Bronze","BRONZE","silver","gold"]),
            "country": fake.country(),
            "signup_date": fake.date()
        })
    pd.DataFrame(customers).to_csv(os.path.join(out,"customers.csv"), index=False)

    # Orders
    orders = []
    for i in range(rows):
        orders.append({
            "order_id": f"O{i}",
            "customer_id": np.random.choice([f"C{j}" for j in range(rows)] + [None]),
            "order_date": np.random.choice([fake.date(), fake.date(pattern="%d/%m/%Y")]),
            "status": np.random.choice(["completed","pending","cancelled"]),
            "total_amount": np.random.choice([round(np.random.uniform(-100,500),2), None]),
            "discount_pct": round(np.random.uniform(0,30),2)
        })
    pd.DataFrame(orders).to_csv(os.path.join(out,"orders.csv"), index=False)

    # Order items
    items = []
    for i in range(rows*2):
        items.append({
            "order_id": np.random.choice([f"O{j}" for j in range(rows)] + ["X999"]), # some orphaned
            "product_id": f"P{i}",
            "category": np.random.choice(["electronics","fashion","books","toys"]),
            "quantity": np.random.randint(1,5),
            "unit_price": round(np.random.uniform(5,100),2)
        })
    pd.DataFrame(items).to_csv(os.path.join(out,"order_items.csv"), index=False)

    # Returns
    returns = []
    for i in range(int(rows*0.3)):
        returns.append({
            "return_id": f"R{i}",
            "order_id": np.random.choice([f"O{j}" for j in range(rows)]),
            "reason": np.random.choice(["damaged","wrong item","other"]),
            "refund_amount": round(np.random.uniform(10,600),2)
        })
    pd.DataFrame(returns).to_csv(os.path.join(out,"returns.csv"), index=False)

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--rows", type=int, default=2000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out", type=str, default="./data")
    args = parser.parse_args()

    os.makedirs(args.out, exist_ok=True)
    generate_data(args.rows, args.seed, args.out)


Overwriting generate_data.py


In [20]:
!python generate_data.py --rows 2000 --seed 42 --out ./data


In [21]:
!ls ./data


customers.csv  order_items.csv	orders.csv  returns.csv


In [22]:
from pyspark.sql import SparkSession, types as T

# 1. Start Spark
spark = SparkSession.builder.appName("ECommercePipeline").getOrCreate()

# 2. Define schemas
orders_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("order_date", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("total_amount", T.DoubleType(), True),
    T.StructField("discount_pct", T.DoubleType(), True)
])

customers_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("customer_tier", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("signup_date", T.StringType(), True)
])

order_items_schema = T.StructType([
    T.StructField("order_id", T.StringType(), True),
    T.StructField("product_id", T.StringType(), True),
    T.StructField("category", T.StringType(), True),
    T.StructField("quantity", T.IntegerType(), True),
    T.StructField("unit_price", T.DoubleType(), True)
])

returns_schema = T.StructType([
    T.StructField("return_id", T.StringType(), True),
    T.StructField("order_id", T.StringType(), True),
    T.StructField("reason", T.StringType(), True),
    T.StructField("refund_amount", T.DoubleType(), True)
])

# 3. Load CSVs with schemas
orders = spark.read.csv("./data/orders.csv", header=True, schema=orders_schema)
customers = spark.read.csv("./data/customers.csv", header=True, schema=customers_schema)
order_items = spark.read.csv("./data/order_items.csv", header=True, schema=order_items_schema)
returns = spark.read.csv("./data/returns.csv", header=True, schema=returns_schema)

# Quick check
orders.show(5)
customers.show(5)


+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
|      O0|       C987|2016-09-08|cancelled|        NULL|       18.96|
|      O1|       C440|01/07/1975|completed|      492.67|       17.91|
|      O2|      C1681|1981-09-23|  pending|      156.92|       21.74|
|      O3|       C458|2025-05-14|  pending|        NULL|       10.12|
|      O4|       C287|1976-07-18|completed|        NULL|       19.36|
+--------+-----------+----------+---------+------------+------------+
only showing top 5 rows
+-----------+-------------+--------------------+-----------+
|customer_id|customer_tier|             country|signup_date|
+-----------+-------------+--------------------+-----------+
|         C0|       silver|Northern Mariana ...| 1976-04-23|
|         C1|         gold|Saint Vincent and...| 1985-08-02|
|         C2|       BRONZE|      Czech Re

In [23]:
from pyspark.sql import functions as F

def clean_orders(orders):
    # 1. Remove duplicates
    orders = orders.dropDuplicates()

    # 2. Normalize dates to ISO (YYYY-MM-DD)
    # Handles both YYYY-MM-DD and DD/MM/YYYY
    orders = orders.withColumn(
        "order_date",
        F.to_date(
            F.when(F.col("order_date").rlike(r"\d{2}/\d{2}/\d{4}"),
                   F.regexp_replace("order_date", r"(\d{2})/(\d{2})/(\d{4})", r"$3-$2-$1"))
             .otherwise(F.col("order_date")),
            "yyyy-MM-dd"
        )
    )

    # 3. Drop rows with NULL order_id or customer_id
    orders = orders.dropna(subset=["order_id", "customer_id"])

    # 4. Flag negative amounts
    orders = orders.withColumn("is_negative_amount", F.col("total_amount") < 0)

    return orders


def clean_customers(customers):
    # Remove duplicates
    customers = customers.dropDuplicates()

    # Normalize signup_date
    customers = customers.withColumn(
        "signup_date",
        F.to_date(
            F.when(F.col("signup_date").rlike(r"\d{2}/\d{2}/\d{4}"),
                   F.regexp_replace("signup_date", r"(\d{2})/(\d{2})/(\d{4})", r"$3-$2-$1"))
             .otherwise(F.col("signup_date")),
            "yyyy-MM-dd"
        )
    )

    # Standardize customer_tier to lowercase
    customers = customers.withColumn("customer_tier", F.lower(F.col("customer_tier")))

    return customers


def clean_order_items(order_items):
    # Remove duplicates
    return order_items.dropDuplicates()


def clean_returns(returns):
    # Remove duplicates
    returns = returns.dropDuplicates()

    # Normalize refund_amount anomalies will be flagged later in Task 05
    return returns


In [24]:
orders = clean_orders(orders)
customers = clean_customers(customers)
order_items = clean_order_items(order_items)
returns = clean_returns(returns)

orders.show(5)
customers.show(5)


+--------+-----------+----------+---------+------------+------------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+---------+------------+------------+------------------+
|    O475|       C935|1999-08-15|cancelled|       19.55|       16.14|             false|
|    O480|       C829|1975-04-13|completed|      433.93|        5.26|             false|
|    O733|       C236|2024-10-21|cancelled|      330.48|        6.53|             false|
|    O907|       C343|2004-06-13|cancelled|       -91.8|       10.41|              true|
|   O1289|       C957|2003-10-16|completed|       218.8|       29.07|             false|
+--------+-----------+----------+---------+------------+------------+------------------+
only showing top 5 rows
+-----------+-------------+--------------------+-----------+
|customer_id|customer_tier|             country|signup_date|
+-----------+-------------+--------------------+-----

In [25]:
def enrich_data(orders, customers, order_items):
    # 1. Join orders with customers (left join so we keep all orders)
    enriched = orders.join(customers, "customer_id", "left")

    # 2. Join order_items with orders (inner join, only valid matches)
    enriched = enriched.join(order_items, "order_id", "inner")

    # 3. Anti-join to isolate orphaned order_items (items with no matching order)
    orphaned_items = order_items.join(orders, "order_id", "anti")
    orphaned_items.write.csv("./output/orphaned_items.csv", mode="overwrite", header=True)

    # 4. Derived column: net_amount = total_amount × (1 - discount_pct/100)
    enriched = enriched.withColumn(
        "net_amount",
        F.col("total_amount") * (1 - F.col("discount_pct") / 100)
    )

    return enriched


In [26]:
enriched = enrich_data(orders, customers, order_items)
enriched.show(5)


+--------+-----------+----------+---------+------------+------------+------------------+-------------+---------------+-----------+----------+-----------+--------+----------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|customer_tier|        country|signup_date|product_id|   category|quantity|unit_price|        net_amount|
+--------+-----------+----------+---------+------------+------------+------------------+-------------+---------------+-----------+----------+-----------+--------+----------+------------------+
|    O475|       C935|1999-08-15|cancelled|       19.55|       16.14|             false|       bronze|Solomon Islands| 2026-01-02|     P2432|       toys|       1|     83.13|          16.39463|
|    O480|       C829|1975-04-13|completed|      433.93|        5.26|             false|       bronze|          Korea| 1975-11-13|     P1613|electronics|       3|     77.38|        411.105282|
|    O733|       C236|2024-10-21|ca

In [27]:
from pyspark.sql import Window

def run_aggregations(enriched):
    # 6. Customers ranked by lifetime net spend within each country
    spend_df = enriched.groupBy("customer_id", "country") \
        .agg(F.sum("net_amount").alias("lifetime_spend"))

    w_rank = Window.partitionBy("country").orderBy(F.desc("lifetime_spend"))
    ranked_customers = spend_df.withColumn("rank", F.rank().over(w_rank))

    # 7. 7-day rolling order count per customer
    # Order_date must be a proper date column from Task 02
    w_roll = Window.partitionBy("customer_id").orderBy("order_date").rangeBetween(-7, 0)
    rolling_orders = enriched.withColumn("rolling_orders", F.count("order_id").over(w_roll))

    # 8. Each product category's share of total revenue per calendar month
    monthly_revenue = enriched.withColumn("month", F.month("order_date")) \
        .groupBy("category", "month") \
        .agg(F.sum("net_amount").alias("category_revenue"))

    total_revenue = monthly_revenue.groupBy("month") \
        .agg(F.sum("category_revenue").alias("total_revenue"))

    category_share = monthly_revenue.join(total_revenue, "month") \
        .withColumn("share", F.col("category_revenue") / F.col("total_revenue"))

    return ranked_customers, rolling_orders, category_share


In [28]:
from pyspark.sql import functions as F, Window

def run_aggregations(enriched):
    # 6. Customers ranked by lifetime net spend within each country
    spend_df = enriched.groupBy("customer_id", "country") \
        .agg(F.sum("net_amount").alias("lifetime_spend"))

    w_rank = Window.partitionBy("country").orderBy(F.desc("lifetime_spend"))
    ranked_customers = spend_df.withColumn("rank", F.rank().over(w_rank))

    # 7. 7-day rolling order count per customer
    # Convert order_date to days since epoch for rangeBetween to work with BIGINT type
    enriched_with_days = enriched.withColumn(
        "order_date_days",
        F.datediff(F.col("order_date"), F.lit("1970-01-01"))
    )

    # Now use the 'order_date_days' (BIGINT) for the window ordering and range
    w_roll = Window.partitionBy("customer_id").orderBy("order_date_days").rangeBetween(-7, 0)
    rolling_orders = enriched_with_days.withColumn("rolling_orders", F.count("order_id").over(w_roll))

    # 8. Each product category's share of total revenue per calendar month
    monthly_revenue = enriched.withColumn("month", F.month("order_date")) \
        .groupBy("category", "month") \
        .agg(F.sum("net_amount").alias("category_revenue"))

    total_revenue = monthly_revenue.groupBy("month") \
        .agg(F.sum("category_revenue").alias("total_revenue"))

    category_share = monthly_revenue.join(total_revenue, "month") \
        .withColumn("share", F.col("category_revenue") / F.col("total_revenue"))

    return ranked_customers, rolling_orders, category_share

ranked_customers, rolling_orders, category_share = run_aggregations(enriched)

ranked_customers.show(5)
rolling_orders.show(5)
category_share.show(5)

+-----------+-----------+------------------+----+
|customer_id|    country|    lifetime_spend|rank|
+-----------+-----------+------------------+----+
|      C1782|Afghanistan|         416.88304|   1|
|        C82|Afghanistan|407.73700299999996|   2|
|       C929|Afghanistan|360.77698799999996|   3|
|      C1657|Afghanistan|290.85617399999995|   4|
|       C191|Afghanistan|              NULL|   5|
+-----------+-----------+------------------+----+
only showing top 5 rows
+--------+-----------+----------+---------+------------+------------+------------------+-------------+--------------------+-----------+----------+-----------+--------+----------+------------------+---------------+--------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|customer_tier|             country|signup_date|product_id|   category|quantity|unit_price|        net_amount|order_date_days|rolling_orders|
+--------+-----------+----------+---------+------------+------------

In [29]:
def return_analysis(enriched, returns):
    # 1. Join returns with enriched orders
    joined = returns.join(enriched, "order_id", "inner")

    # 2. Compute return rate (returns / orders) per category and per customer_tier
    return_rate = joined.groupBy("category", "customer_tier") \
        .agg((F.count("return_id") / F.countDistinct("order_id")).alias("return_rate"))

    # 3. Identify top 10 customers by total refund amount
    top_refunds = joined.groupBy("customer_id") \
        .agg(F.sum("refund_amount").alias("total_refund")) \
        .orderBy(F.desc("total_refund")) \
        .limit(10)

    # 4. Flag anomalies: refund_amount > net_amount
    joined = joined.withColumn("refund_exceeds_order", F.col("refund_amount") > F.col("net_amount"))

    return return_rate, top_refunds, joined


In [30]:
return_rate, top_refunds, returns_enriched = return_analysis(enriched, returns)

return_rate.show(5)
top_refunds.show(5)
returns_enriched.select("order_id","refund_amount","net_amount","refund_exceeds_order").show(5)


+-----------+-------------+------------------+
|   category|customer_tier|       return_rate|
+-----------+-------------+------------------+
|    fashion|       bronze|1.4710743801652892|
|       toys|       bronze|1.4017857142857142|
|       toys|         gold|              1.52|
|electronics|       bronze|1.4770642201834863|
|       toys|       silver|              1.48|
+-----------+-------------+------------------+
only showing top 5 rows
+-----------+-----------------+
|customer_id|     total_refund|
+-----------+-----------------+
|       C661|9293.359999999999|
|       C232|6583.500000000001|
|       C667|5851.110000000001|
|      C1848|          5215.08|
|      C1579|          4532.97|
+-----------+-----------------+
only showing top 5 rows
+--------+-------------+------------------+--------------------+
|order_id|refund_amount|        net_amount|refund_exceeds_order|
+--------+-------------+------------------+--------------------+
|    O693|       327.37|15.989480999999998|   

In [31]:
def write_outputs(enriched, ranked_customers, rolling_orders, category_share):
    # Add year/month columns for partitioning
    enriched = enriched.withColumn("order_year", F.year("order_date")) \
                       .withColumn("order_month", F.month("order_date"))

    # 1. Write enriched dataset to Parquet, partitioned by year/month
    enriched.write.partitionBy("order_year", "order_month") \
        .mode("overwrite") \
        .parquet("./output/enriched")

    # 2. Write aggregated summary tables to CSV
    ranked_customers.write.csv("./output/ranked_customers.csv", mode="overwrite", header=True)
    rolling_orders.write.csv("./output/rolling_orders.csv", mode="overwrite", header=True)
    category_share.write.csv("./output/category_share.csv", mode="overwrite", header=True)


In [32]:
write_outputs(enriched, ranked_customers, rolling_orders, category_share)
